# Stage 00a.1 — Dataset Audit

**What this notebook does:**  
Audits the `raw/` folder against the PatSeer Excel export and produces:

| Output | Description |
|--------|-------------|
| `1639_DS_audit.xlsx` | Full per-patent status: matched, missing, extra, name mismatches, partial downloads |
| `patseer_missing_prompt.txt` | Ready-to-paste `PNC:(...)` queries for PatSeer re-search |
| `disk_inventory.xlsx` | Per-folder image provenance check (pub-number prefix vs expected family) |
| `patseer_expected_inventory.xlsx` | Per-Excel-row: folder exists, image count, provenance result |
| `provenance_fail_detail.csv` | Detail on folders whose images belong to the wrong patent |
| `provenance_unknown_not_flagged.csv` | fig_NN folders not yet in the Needs Redownload list (gaps) |

**Statuses assigned to each Excel record:**

| Status | Meaning |
|--------|---------|
| `OK` | Folder found, all images properly named |
| `OK – prefixed fig_NN` | Folder found with `{pub}_fig_NN.png` names — valid PatSeer naming for some patents, not an error |
| `NEEDS REDOWNLOAD – partial` | Folder found but some images are bare `fig_NN.png` (no patent prefix) — download was interrupted by 400 errors |
| `MISSING from folder` | No folder found — needs to be downloaded |
| `DUPLICATE folders` | Two or more folders normalize to the same patent ID |

**Name normalization rules** (applied to both Excel IDs and folder names):
- Strip trailing `/`, `PAFP` suffix
- **US publications** (`US19xx`, `US20xx`): `US2022267016A1` → `US2022_267016` (handles leading-zero variants)
- **US grants** (no year prefix, e.g. `US11787551B1`): strip kind code only → `US11787551`
- **USD design patents**: `USD902828S` → `SD902828` (PatSeer drops the `U` prefix)
- All others: strip kind codes (`A1`, `B2`, `C1`, `D0`, `S1`, `U1` …) → base number
- `record_XXXX` identifiers are kept as-is

**Where it fits in the pipeline:**
```
00a.1  ← YOU ARE HERE  (audit what is downloaded vs what is in Excel)
 ↓
00a    (download missing patents from PatSeer using generated PNC query)
 ↓
00b    (positional matching → _F / _Fu labels)
```

---

| Cell | What it does |
|------|--------------|
| 1 | Load config and paths |
| 2 | Run audit — compare Excel vs `raw/` folders, detect partial downloads |
| 3 | Write `1639_DS_audit.xlsx` |
| 4 | Print summary + generate PatSeer PNC prompt for missing patents |
| 5 | Verify folder naming convention (`{record_number}_{family_id}`) |
| 6 | Disk inventory — extract pub-number prefix from every image filename |
| 7 | PatSeer expected inventory — join Excel rows to folders + image counts |
| 8 | Cross-reference integrity report |
| 9 | **Provenance deep-dive** — detail on FAIL folders + UNKNOWN gap check |
| 10 | **Rename legacy folders** — renames old-style folders to `{pub}_{fam_id}`, quarantines truly-extra ones |

In [1]:
import sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
src_dir   = repo_root / "src"

_cl_path = src_dir / "config_loader.py"
_cl_spec = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod  = importlib.util.module_from_spec(_cl_spec)
sys.modules["config_loader"] = _cl_mod
_cl_spec.loader.exec_module(_cl_mod)
load_config = _cl_mod.load_config

cfg = load_config()

RAW_DIR       = Path(cfg["paths"]["raw_images"])
EXCEL_PATH    = Path(cfg["paths"]["patseer_excel"])
OUTPUT_DIR    = repo_root / "notebooks" / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_EXCEL  = OUTPUT_DIR / "1639_DS_audit.xlsx"
OUTPUT_PROMPT = OUTPUT_DIR / "patseer_missing_prompt.txt"

print(f"Raw folder : {RAW_DIR}")
print(f"Excel      : {EXCEL_PATH}")
print(f"Outputs    : {OUTPUT_DIR}")

Raw folder : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/raw
Excel      : /mnt/storage_11tb/Drive_files_to_syncronize/2 - Patente & Validation/3 -Raw_Patent_Exports_PatSeer_&Gold_Standard/1639__dataset_08_06_26.xlsx
Outputs    : /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/notebooks/outputs


In [2]:


import pandas as pd
df = pd.read_excel(cfg["paths"]["patseer_excel"], dtype=str)
fam = df["Simple Family ID"].dropna().str.strip()
print(f"Total rows:          {len(df)}")
print(f"Unique Family IDs:   {fam.nunique()}")
print(f"Blank Family IDs:    {(df['Simple Family ID'].isna() | (df['Simple Family ID']=='')).sum()}")
print(f"Repeated Family IDs: {fam[fam.duplicated()].nunique()} families with >1 member")


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Total rows:          1639
Unique Family IDs:   1639
Blank Family IDs:    0
Repeated Family IDs: 0 families with >1 member


In [3]:
import re
import json
import pandas as pd
from collections import Counter
from pathlib import Path


def normalize_id(raw: str) -> str:
    pid = str(raw).strip().rstrip("/")
    pid = re.sub(r"PAFP$", "", pid)
    if re.match(r"^record_\d+$", pid):
        return pid
    if re.match(r"^USD\d", pid):
        pid = "SD" + pid[3:]
    us_pub = re.match(r"^(US)((?:19|20)\d{2})0*(\d+)[A-Za-z0-9]*$", pid)
    if us_pub:
        country, year, core = us_pub.groups()
        return f"{country}{year}_{core}"
    pid = re.sub(r"[A-Z][0-9]$", "", pid)
    pid = re.sub(r"[A-Z]$", "", pid)
    return pid


# Folders are named {patent_id}_{family_id} (e.g. US2022267016A1_75223306).
# Extract the numeric family ID suffix for matching against Simple Family ID.
_FAM_SUFFIX_RE = re.compile(r"_(\d+)$")

def _folder_family_id(name: str) -> str:
    """Return the numeric family ID from a {patent_id}_{family_id} folder name."""
    m = _FAM_SUFFIX_RE.search(name)
    return m.group(1) if m else ""


def _scan_folder(folder_path: Path) -> dict:
    pngs = [f for f in folder_path.iterdir() if f.suffix.lower() in (".png", ".jpg", ".jpeg")]
    # A true 400-error fallback has NO patent-ID prefix: bare "fig_01.png".
    # Files like "EP3119674A2_fig_01.png" are valid PatSeer naming — not errors.
    fallbacks = [f for f in pngs if re.match(r"fig_\d+\.png$", f.name, re.IGNORECASE)]
    proper    = [f for f in pngs if f not in fallbacks]
    all_fallback = len(pngs) > 0 and len(proper) == 0
    partial      = len(fallbacks) > 0 and len(proper) > 0
    record_pos = None
    for mf in folder_path.glob("*.json"):
        try:
            data = json.loads(mf.read_text())
            record_pos = data.get("record_position")
            if record_pos:
                break
        except Exception:
            pass
    return {
        "image_count":    len(pngs),
        "has_images":     len(pngs) > 0,
        "fallback_files": sorted(f.name for f in fallbacks),
        "all_fallback":   all_fallback,
        "partial":        partial,
        "record_pos":     record_pos,
    }


# ── Load Excel — index by Simple Family ID ───────────────────────────────────
df_excel    = pd.read_excel(EXCEL_PATH, dtype=str)
id_col      = "Record Number"
fam_col     = "Simple Family ID"
fam_mem_col = "Simple Family Members"

excel_records = []
for i, row in df_excel.iterrows():
    pub = str(row.get(id_col, "")).strip()
    fam = str(row.get(fam_col, "")).strip()
    if not pub or pub.lower() == "nan":
        continue
    members_raw = str(row.get(fam_mem_col, "")).strip()
    member_pubs = [m.strip() for m in re.split(r"[/\n]+", members_raw)
                   if m.strip() and m.strip().lower() != "nan"]
    all_pubs = list({pub} | set(member_pubs))
    excel_records.append({
        "excel_row":     i + 2,
        "record_number": pub,
        "family_id":     fam,
        "all_pub_norms": [normalize_id(p) for p in all_pubs],
        "all_pubs":      all_pubs,
    })

# ── Load raw folders ──────────────────────────────────────────────────────────
# Primary key: numeric family ID extracted from the {patent_id}_{family_id} name
raw_folders = []
for entry in sorted(RAW_DIR.iterdir()):
    if not entry.is_dir():
        continue
    stats = _scan_folder(entry)
    raw_folders.append({
        "folder_name": entry.name,
        "family_id":   _folder_family_id(entry.name),   # numeric suffix
        "norm":        normalize_id(entry.name),
        **stats,
    })

n_record_folders  = sum(1 for f in raw_folders if f["all_fallback"] and f["folder_name"].startswith("record_"))
n_partial_folders = sum(1 for f in raw_folders if f["partial"])
n_zero_imgs       = sum(1 for f in raw_folders if not f["has_images"])

print(f"Excel records (families)     : {len(excel_records)}")
print(f"Raw folders total            : {len(raw_folders)}")
print(f"  → with 0 images           : {n_zero_imgs}")
print(f"  → record_XXXX (fallbacks) : {n_record_folders}")
print(f"  → partial (some fallbacks): {n_partial_folders}")


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Excel records (families)     : 1639
Raw folders total            : 1600
  → with 0 images           : 2
  → record_XXXX (fallbacks) : 0
  → partial (some fallbacks): 0


In [4]:
# ── Build lookup tables ───────────────────────────────────────────────────────
# Primary lookup: numeric family_id suffix → folder entry
fam_id_to_folder = {}
for f in raw_folders:
    if f["family_id"]:
        fam_id_to_folder[f["family_id"]] = f

# Secondary lookup: norm(folder_name) → folder  (fallback for unrecognised names)
raw_norm_map = {}
for f in raw_folders:
    raw_norm_map.setdefault(f["norm"], []).append(f)

# ── Classify each Excel record ────────────────────────────────────────────────
rows_audit       = []
missing_families = []
needs_redownload = []

for r in excel_records:
    fam = r["family_id"]

    # Match 1: numeric family ID suffix in folder name  (primary — new naming)
    matched_folder = fam_id_to_folder.get(fam)

    # Match 2: normalised pub number matches folder name  (legacy fallback)
    if matched_folder is None:
        for norm in r["all_pub_norms"]:
            hits = raw_norm_map.get(norm, [])
            if hits:
                matched_folder = hits[0]
                break

    # Match 3: record_XXXX via record_position
    if matched_folder is None:
        rec_name = f"record_{r['excel_row'] - 1:04d}"
        hits = raw_norm_map.get(rec_name, [])
        if hits:
            matched_folder = hits[0]

    if matched_folder is None:
        status        = "MISSING from folder"
        missing_families.append(r)
        folder_name   = ""
        image_count   = ""
        fallback_info = ""
    else:
        mf = matched_folder
        if mf["all_fallback"]:
            status = "NEEDS REDOWNLOAD – all images are bare fig_NN (400 errors, no patent prefix)"
            needs_redownload.append({
                "family_id":     fam,
                "record_number": r["record_number"],
                "excel_row":     r["excel_row"],
                "folder_name":   mf["folder_name"],
                "image_count":   mf["image_count"],
                "issue":         "all images are bare fig_NN (400 errors — no patent-ID prefix in filename)",
            })
            fallback_info = f"{mf['image_count']} fallback files"
        elif mf["partial"]:
            status = "NEEDS REDOWNLOAD – partial (some bare fig_NN, download interrupted)"
            needs_redownload.append({
                "family_id":     fam,
                "record_number": r["record_number"],
                "excel_row":     r["excel_row"],
                "folder_name":   mf["folder_name"],
                "image_count":   mf["image_count"],
                "issue":         f"partial: {len(mf['fallback_files'])} fallback(s): {' | '.join(mf['fallback_files'][:3])}",
            })
            fallback_info = " | ".join(mf["fallback_files"])
        else:
            status        = "OK"
            fallback_info = ""
        folder_name = mf["folder_name"]
        image_count = str(mf["image_count"])

    rows_audit.append({
        "Excel Row":        r["excel_row"],
        "Simple Family ID": fam,
        "Record Number":    r["record_number"],
        "Status":           status,
        "Matched Folder":   folder_name,
        "Image Count":      image_count,
        "Fallback Files":   fallback_info,
    })

# ── Extra folders (on disk but not matched to any Excel family) ───────────────
matched_folder_names = {r["Matched Folder"] for r in rows_audit if r["Matched Folder"]}
excel_family_ids     = {r["family_id"] for r in excel_records}

extra_folders = [
    {"Folder Name": f["folder_name"], "Image Count": f["image_count"]}
    for f in raw_folders
    if f["family_id"] not in excel_family_ids
    and f["folder_name"] not in matched_folder_names
    and not f["folder_name"].startswith("record_")
]
unresolved_records = [
    {"Folder Name": f["folder_name"], "Image Count": f["image_count"],
     "Note": "could not identify family"}
    for f in raw_folders
    if f["folder_name"].startswith("record_")
    and f["folder_name"] not in matched_folder_names
]

raw_norm_counts = Counter(f["norm"] for f in raw_folders)
raw_dupes = [
    {"Folder Name": f["folder_name"], "Image Count": f["image_count"], "Normalized Key": f["norm"]}
    for f in raw_folders if raw_norm_counts[f["norm"]] > 1
]

print("Classification done.")
status_counts = Counter(r["Status"] for r in rows_audit)
for s, c in sorted(status_counts.items()):
    print(f"  {s}: {c}")
print(f"  Extra folders (not in Excel) : {len(extra_folders)}")
print(f"  Unresolved record_ folders   : {len(unresolved_records)}")
print(f"  Duplicate norm keys in raw/  : {len(raw_dupes)}")


Classification done.
  MISSING from folder: 46
  OK: 1593
  Extra folders (not in Excel) : 7
  Unresolved record_ folders   : 0
  Duplicate norm keys in raw/  : 2


In [5]:
df_audit      = pd.DataFrame(rows_audit)
df_extra      = pd.DataFrame(extra_folders)      if extra_folders      else pd.DataFrame()
df_unresolved = pd.DataFrame(unresolved_records) if unresolved_records else pd.DataFrame()
df_rdupes     = pd.DataFrame(raw_dupes)          if raw_dupes          else pd.DataFrame()
df_redownload = pd.DataFrame(needs_redownload)   if needs_redownload   else pd.DataFrame()

n_zero_imgs = sum(1 for f in raw_folders if not f["has_images"])

summary_rows = [
    ("Total Excel families",                               len(excel_records)),
    ("Total raw folders",                                 len(raw_folders)),
    ("  → Folders with 0 images",                        n_zero_imgs),
    ("  → record_XXXX (all images fallbacks)",           n_record_folders),
    ("  → Partial (some images fallbacks)",              n_partial_folders),
    ("",                                                  ""),
    ("OK (matched, clean)",                               status_counts.get("OK", 0)),
    ("NEEDS REDOWNLOAD – all bare fig_NN (400 errors)",   status_counts.get("NEEDS REDOWNLOAD – all images are bare fig_NN (400 errors, no patent prefix)", 0)),
    ("NEEDS REDOWNLOAD – partial bare fig_NN",            status_counts.get("NEEDS REDOWNLOAD – partial (some bare fig_NN, download interrupted)", 0)),
    ("MISSING from folder",                               status_counts.get("MISSING from folder", 0)),
    ("",                                                  ""),
    ("Extra folders (not in Excel)",                      len(extra_folders)),
    ("Unresolved record_ folders",                        len(unresolved_records)),
    ("Duplicate norm keys in raw folders",                len(raw_dupes)),
]
df_summary = pd.DataFrame([(m, c) for m, c in summary_rows], columns=["Metric", "Count"])

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    df_summary.to_excel(   writer, sheet_name="Summary",            index=False)
    df_audit.to_excel(     writer, sheet_name="Full Audit",         index=False)
    if not df_redownload.empty:
        df_redownload.to_excel(writer, sheet_name="Needs Redownload",   index=False)
    if not df_extra.empty:
        df_extra.to_excel( writer, sheet_name="Extra Folders",      index=False)
    if not df_unresolved.empty:
        df_unresolved.to_excel(writer, sheet_name="Unresolved record_", index=False)
    if not df_rdupes.empty:
        df_rdupes.to_excel(writer, sheet_name="Raw Folder Duplicates",  index=False)

print(f"Audit Excel written to: {OUTPUT_EXCEL}")


Audit Excel written to: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/notebooks/outputs/1639_DS_audit.xlsx


In [6]:
# ── Print summary ─────────────────────────────────────────────────────────────
print("══════════════════════════════════════════════════════════")
print("Dataset Audit Summary")
print("══════════════════════════════════════════════════════════")
for metric, count in summary_rows:
    if metric == "":
        print()
        continue
    print(f"  {metric:<52}: {count}")
print("══════════════════════════════════════════════════════════")

if needs_redownload:
    print(f"\n  Needs redownload (first 10 of {len(needs_redownload)}):")
    for p in needs_redownload[:10]:
        print(f"    Row {p['excel_row']:4d}  fam={p['family_id']:12s}  {p['record_number']:30s}  → {p['issue'][:60]}")


def _build_pnc_terms(family_list: list) -> list[str]:
    """One PNC term per family — canonical Record Number, kind-code stripped."""
    seen: set[str] = set()
    terms = []
    for r in family_list:
        for pub in r["all_pubs"]:
            clean = re.sub(r"PAFP$", "", pub.strip())
            clean = re.sub(r"[A-Z][0-9]$", "", clean).strip()
            if clean and clean.lower() not in ("nan", "") and clean not in seen:
                seen.add(clean)
                terms.append(f'"{clean}"')
                break
    return terms


def _write_pnc_prompt(out_path, family_list: list, label: str, extra_note: str = "") -> None:
    terms  = _build_pnc_terms(family_list)
    CHUNK  = 350
    chunks = [terms[i:i+CHUNK] for i in range(0, len(terms), CHUNK)]
    lines  = [
        f"# PatSeer PNC Query — {len(family_list)} {label} families ({len(terms)} terms)",
        f"# Split into {len(chunks)} chunk(s) of up to {CHUNK} terms each",
    ]
    if extra_note:
        lines += ["#", f"# {extra_note}"]
    lines.append("")
    for i, chunk in enumerate(chunks, 1):
        lines.append(f"# --- Chunk {i} ({len(chunk)} terms) ---")
        lines.append("PNC:(" + " OR ".join(chunk) + ")\n")
    out_path.write_text("\n".join(lines))
    print(f"  → {out_path}")
    print(f"     {len(family_list)} families, {len(chunks)} chunk(s)")
    if terms:
        print(f"     Preview: PNC:({' OR '.join(terms[:3])} OR ...)")


import csv

# ── Build family_id → excel_record lookup (needed for redownload PNC) ────────
fam_id_to_record = {r["family_id"]: r for r in excel_records}

# ── MISSING families ──────────────────────────────────────────────────────────
print()
missing_csv = OUTPUT_DIR / "missing_families.csv"
with missing_csv.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["family_id", "record_number", "excel_row", "all_pubs"])
    writer.writeheader()
    for r in missing_families:
        writer.writerow({
            "family_id":     r["family_id"],
            "record_number": r["record_number"],
            "excel_row":     r["excel_row"],
            "all_pubs":      " | ".join(r["all_pubs"]),
        })

# ── NEEDS REDOWNLOAD families (as excel_records format for PNC builder) ───────
redownload_records = [
    fam_id_to_record[p["family_id"]]
    for p in needs_redownload
    if p["family_id"] in fam_id_to_record
]

# ── COMBINED prompt: missing + needs redownload → patseer_missing_prompt.txt ─
# Always written so the file is never stale from a previous run.
all_to_download = missing_families + redownload_records
print(f"PatSeer prompt (COMBINED: {len(missing_families)} missing + {len(redownload_records)} redownload = {len(all_to_download)} families):")
_write_pnc_prompt(
    OUTPUT_PROMPT,   # → patseer_missing_prompt.txt
    all_to_download,
    label="MISSING + NEEDS REDOWNLOAD",
    extra_note=(
        "Run 00a Cell 1b first to quarantine the bad fig_NN folders, "
        "then use this query in PatSeer and download via 00a Cell 3."
    ),
)
if missing_families:
    print(f"     CSV (missing only) → {OUTPUT_DIR / 'missing_families.csv'}")
if not all_to_download:
    print("     (empty — all families are present and clean)")

# ── Separate per-category prompts (for reference / partial runs) ──────────────
print()
print(f"PatSeer prompt (MISSING only — {len(missing_families)} families):")
_write_pnc_prompt(
    OUTPUT_DIR / "patseer_missing_only_prompt.txt",
    missing_families,
    label="MISSING only",
)
if not missing_families:
    print("     (empty — no missing families)")

print()
print(f"PatSeer prompt (NEEDS REDOWNLOAD only — {len(redownload_records)} families):")
_write_pnc_prompt(
    OUTPUT_DIR / "patseer_redownload_prompt.txt",
    redownload_records,
    label="NEEDS REDOWNLOAD only",
    extra_note="Quarantine the existing folder via 00a Cell 1b before using this query.",
)
if not redownload_records:
    print("     (empty — no folders need redownload)")

# ── NON-MISSING families prompt ───────────────────────────────────────────────
non_missing = [r for r in excel_records if r["family_id"] not in {m["family_id"] for m in missing_families}]
OUTPUT_NON_MISSING_PROMPT = OUTPUT_DIR / "patseer_non_missing_prompt.txt"
print()
print(f"PatSeer prompt (NON-MISSING — {len(non_missing)} families):")
_write_pnc_prompt(
    OUTPUT_NON_MISSING_PROMPT,
    non_missing,
    label="NON-MISSING (already on disk)",
)

# ── Action summary ─────────────────────────────────────────────────────────────
print(f"""
╔══════════════════════════════════════════════════════════╗
  ACTION PLAN
  ─────────────────────────────────────────────────────
  1. Run 00a Cell 1b to quarantine {len(redownload_records):3d} NEEDS REDOWNLOAD folders
  2. Search PatSeer with patseer_missing_prompt.txt
     ({len(all_to_download)} families: {len(missing_families)} missing + {len(redownload_records)} redownload)
  3. Run 00a Cell 3 to download the results
  4. {status_counts.get("OK", 0):4d} families are clean ✓
╚══════════════════════════════════════════════════════════╝
""")


══════════════════════════════════════════════════════════
Dataset Audit Summary
══════════════════════════════════════════════════════════
  Total Excel families                                : 1639
  Total raw folders                                   : 1600
    → Folders with 0 images                           : 2
    → record_XXXX (all images fallbacks)              : 0
    → Partial (some images fallbacks)                 : 0

  OK (matched, clean)                                 : 1593
  NEEDS REDOWNLOAD – all bare fig_NN (400 errors)     : 0
  NEEDS REDOWNLOAD – partial bare fig_NN              : 0
  MISSING from folder                                 : 46

  Extra folders (not in Excel)                        : 7
  Unresolved record_ folders                          : 0
  Duplicate norm keys in raw folders                  : 2
══════════════════════════════════════════════════════════

PatSeer prompt (COMBINED: 46 missing + 0 redownload = 46 families):
  → /home/vasco/Vasco Wo

In [7]:
# ── Cell 5: Verify folder naming convention ───────────────────────────────────
# Folders are expected to follow {record_number}_{family_id} (e.g. US2022267016A1_75223306).
# This cell checks every folder conforms and reports any that don't.
import re, sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
_cl_path  = repo_root / "src" / "config_loader.py"
_cl_spec  = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod   = importlib.util.module_from_spec(_cl_spec)
_cl_spec.loader.exec_module(_cl_mod)
cfg = _cl_mod.load_config()

import pandas as pd

raw_dir  = Path(cfg["paths"]["raw_images"])
data_dir = Path(cfg["paths"]["data"])
data_dir.mkdir(parents=True, exist_ok=True)

df_xl   = pd.read_excel(cfg["paths"]["patseer_excel"], dtype=str)
id_col  = "Record Number"
fam_col = "Simple Family ID"

# Build expected folder name: {record_number}_{family_id}
expected: dict[str, str] = {}   # family_id → expected folder name
for _, row in df_xl.iterrows():
    pub = str(row.get(id_col, "")).strip()
    fam = str(row.get(fam_col, "")).strip()
    if pub and fam and pub.lower() != "nan" and fam.lower() != "nan":
        expected[fam] = f"{pub}_{fam}"

_FAM_SUFFIX_RE_5 = re.compile(r"_(\d+)$")

ok = wrong_format = not_in_excel = 0
issues = []

for folder in sorted(raw_dir.iterdir()):
    if not folder.is_dir():
        continue
    name = folder.name
    m    = _FAM_SUFFIX_RE_5.search(name)
    if not m:
        wrong_format += 1
        issues.append({"folder": name, "issue": "no numeric family_id suffix"})
        continue
    fam_id       = m.group(1)
    expected_name = expected.get(fam_id, "")
    if not expected_name:
        not_in_excel += 1
        issues.append({"folder": name, "issue": "family_id not in Excel"})
    elif name != expected_name:
        wrong_format += 1
        issues.append({"folder": name, "issue": f"expected '{expected_name}'"})
    else:
        ok += 1

print(f"Folders checked   : {ok + wrong_format + not_in_excel}")
print(f"  ✓ Correctly named : {ok}")
print(f"  ✗ Wrong format    : {wrong_format}")
print(f"  ? Not in Excel    : {not_in_excel}")

if issues:
    import csv
    issues_path = data_dir / "folder_naming_issues.csv"
    with issues_path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["folder", "issue"])
        w.writeheader()
        w.writerows(issues)
    print(f"\nIssues log → {issues_path}")
    for row in issues[:10]:
        print(f"  {row['folder']}  →  {row['issue']}")
    if len(issues) > 10:
        print(f"  … and {len(issues) - 10} more (see CSV)")
else:
    print("\n✓ All folders correctly named.")


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Folders checked   : 1600
  ✓ Correctly named : 1415
  ✗ Wrong format    : 185
  ? Not in Excel    : 0

Issues log → /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/folder_naming_issues.csv
  AU2020100605A4  →  no numeric family_id suffix
  CA2947683A1  →  no numeric family_id suffix
  CA2958445A1  →  no numeric family_id suffix
  CA3096221  →  no numeric family_id suffix
  CN112478154A  →  no numeric family_id suffix
  CN114435591A  →  no numeric family_id suffix
  CN117842351A  →  no numeric family_id suffix
  CN119218433A  →  no numeric family_id suffix
  CN120024492A  →  no numeric family_id suffix
  CN120735960A  →  no numeric family_id suffix
  … and 175 more (see CSV)


In [8]:
# ── Cell 6: Disk inventory ────────────────────────────────────────────────────
import re, sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
_cl_path  = repo_root / "src" / "config_loader.py"
_cl_spec  = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod   = importlib.util.module_from_spec(_cl_spec)
_cl_spec.loader.exec_module(_cl_mod)
cfg = _cl_mod.load_config()

import pandas as pd

raw_dir  = Path(cfg["paths"]["raw_images"])
data_dir = Path(cfg["paths"]["data"])
data_dir.mkdir(parents=True, exist_ok=True)

def _normalize_id_6(raw: str) -> str:
    pid = str(raw).strip().rstrip("/")
    pid = re.sub(r"PAFP$", "", pid)
    if re.match(r"^record_\d+$", pid): return pid
    if re.match(r"^USD\d", pid): pid = "SD" + pid[3:]
    us_pub = re.match(r"^(US)((?:19|20)\d{2})0*(\d+)[A-Za-z0-9]*$", pid)
    if us_pub:
        country, year, core = us_pub.groups()
        return f"{country}{year}_{core}"
    pid = re.sub(r"[A-Z][0-9]$", "", pid)
    pid = re.sub(r"[A-Z]$", "", pid)
    return pid

_FAM_SUFFIX_RE_6 = re.compile(r"_(\d+)$")

def _fam_id_6(name: str) -> str:
    m = _FAM_SUFFIX_RE_6.search(name)
    return m.group(1) if m else ""

df_xl   = pd.read_excel(cfg["paths"]["patseer_excel"], dtype=str)
id_col  = "Record Number"
fam_col = "Simple Family ID"
mem_col = "Simple Family Members"

fam_to_pub_6:  dict[str, str] = {}
fam_pub_norms: dict[str, set] = {}
norm_to_fam_6: dict[str, str] = {}
for _, row in df_xl.iterrows():
    pub = str(row.get(id_col, "")).strip()
    fam = str(row.get(fam_col, "")).strip()
    if not pub or not fam or pub.lower() == "nan" or fam.lower() == "nan":
        continue
    fam_to_pub_6.setdefault(fam, pub)
    members_raw = str(row.get(mem_col, "")).strip()
    member_pubs = [m.strip() for m in re.split(r"[/\n]+", members_raw)
                   if m.strip() and m.strip().lower() != "nan"]
    norms = {_normalize_id_6(p) for p in {pub} | set(member_pubs)}
    fam_pub_norms.setdefault(fam, set()).update(norms)
    for n in norms:
        norm_to_fam_6.setdefault(n, fam)

known_families = set(fam_to_pub_6.keys())

# ── Pub-number extraction from filenames ──────────────────────────────────────
# PatSeer produces several filename conventions; we recognise all of them:
#   1. {PUB}_{suffix}.png         e.g. US2022267016A1_img003.png  (standard)
#   2. {PUB}PAFP.png / {PUB}PAFP_{suffix}.png  (PAFP preview image)
#   3. imgNNNN.png / img_NNNN.png               (BR/KR style — no pub in name)
#   4. NNNNNNNNNN.png                           (CN numeric timestamp style)
#   5. fig_NN.png                               (400-error fallback — UNKNOWN)
#
# For cases 3 & 4 provenance is inferred from the PAFP file that always
# accompanies them: if {FOLDER_PUB}PAFP.png exists the download is correct.

_PUB_UNDERSCORE_RE = re.compile(r"^([A-Z]{2}[0-9A-Z]+)_")   # case 1
_PUB_PAFP_RE       = re.compile(r"^([A-Z]{2}[0-9A-Z]+)PAFP") # case 2
_FIG_NN_RE         = re.compile(r"^fig_\d+\.png$", re.IGNORECASE)

def _extract_pub_prefixes(filenames: list[str]) -> set[str]:
    """Return all pub-number strings identifiable from a list of filenames."""
    out: set[str] = set()
    for fn in filenames:
        m = _PUB_UNDERSCORE_RE.match(fn)
        if m:
            out.add(m.group(1).upper())
            continue
        m = _PUB_PAFP_RE.match(fn)
        if m:
            out.add(m.group(1).upper())
    return out

def _has_pafp_for_folder(folder_pub: str, filenames: list[str]) -> bool:
    """True if any file is {folder_pub}PAFP.png (case-insensitive)."""
    target = folder_pub.upper() + "PAFP"
    return any(fn.upper().startswith(target) for fn in filenames)

rows_inv = []
for folder in sorted(raw_dir.iterdir()):
    if not folder.is_dir():
        continue
    pngs = sorted(f.name for f in folder.iterdir() if f.suffix.lower() == ".png")
    image_count      = len(pngs)
    filenames_sample = " | ".join(pngs[:3])

    folder_name = folder.name
    family_id   = _fam_id_6(folder_name)
    if not family_id:
        family_id = norm_to_fam_6.get(_normalize_id_6(folder_name), "")
    expected_pub = fam_to_pub_6.get(family_id, "")

    pub_prefixes      = _extract_pub_prefixes(pngs)
    pub_nums_in_files = " | ".join(sorted(pub_prefixes))

    # ── Provenance decision ───────────────────────────────────────────────────
    if family_id not in known_families:
        provenance_ok = "UNKNOWN"
    elif not pngs:
        provenance_ok = "UNKNOWN"
    else:
        prefix_norms = {_normalize_id_6(p) for p in pub_prefixes}
        family_norms = fam_pub_norms.get(family_id, set())

        if prefix_norms & family_norms:
            # Case 1 or 2: pub number found in filenames and matches family
            provenance_ok = "True"
        elif not pub_prefixes and expected_pub and _has_pafp_for_folder(expected_pub, pngs):
            # Cases 3 & 4: no underscore-prefixed pub in filenames, but the
            # PAFP companion file confirms the correct patent was downloaded
            provenance_ok = "True"
        elif all(_FIG_NN_RE.match(fn) for fn in pngs):
            # Case 5: every file is fig_NN → 400-error fallback, needs redownload
            provenance_ok = "UNKNOWN"
        elif not pub_prefixes:
            # Numeric/imgNNNN files but NO PAFP file → cannot confirm
            provenance_ok = "UNKNOWN"
        else:
            # Pub prefix found but doesn't match any family member
            provenance_ok = "False"

    rows_inv.append({
        "folder_name":        folder_name,
        "family_id":          family_id,
        "image_count":        image_count,
        "filenames_sample":   filenames_sample,
        "pub_nums_in_files":  pub_nums_in_files,
        "expected_pub_num":   expected_pub,
        "provenance_ok":      provenance_ok,
    })

df_inv = pd.DataFrame(rows_inv)
out    = data_dir / "disk_inventory.xlsx"
df_inv.to_excel(out, index=False)

n_ok      = (df_inv["provenance_ok"] == "True").sum()
n_fail    = (df_inv["provenance_ok"] == "False").sum()
n_unknown = (df_inv["provenance_ok"] == "UNKNOWN").sum()

print(f"Disk inventory → {out}")
print(f"  Total folders    : {len(df_inv)}")
print(f"  Provenance OK    : {n_ok}   (pub number confirmed in filenames or via PAFP file)")
print(f"  Provenance FAIL  : {n_fail}   (wrong patent's images)")
print(f"  UNKNOWN          : {n_unknown}   (fig_NN fallbacks — needs redownload)")

/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Disk inventory → /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/disk_inventory.xlsx
  Total folders    : 1600
  Provenance OK    : 1595   (pub number confirmed in filenames or via PAFP file)
  Provenance FAIL  : 0   (wrong patent's images)
  UNKNOWN          : 5   (fig_NN fallbacks — needs redownload)


In [9]:
# ── Cell 7: PatSeer expected inventory ───────────────────────────────────────
import re, sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
_cl_path  = repo_root / "src" / "config_loader.py"
_cl_spec  = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod   = importlib.util.module_from_spec(_cl_spec)
_cl_spec.loader.exec_module(_cl_mod)
cfg = _cl_mod.load_config()

import pandas as pd

raw_dir  = Path(cfg["paths"]["raw_images"])
data_dir = Path(cfg["paths"]["data"])
data_dir.mkdir(parents=True, exist_ok=True)

df_xl   = pd.read_excel(cfg["paths"]["patseer_excel"], dtype=str)
id_col  = "Record Number"
fam_col = "Simple Family ID"

# Load disk inventory for provenance join
inv_path = data_dir / "disk_inventory.xlsx"
if inv_path.exists():
    df_inv   = pd.read_excel(inv_path, dtype=str)
    prov_map = dict(zip(df_inv["family_id"], df_inv["provenance_ok"]))
else:
    prov_map = {}
    print("⚠  disk_inventory.xlsx not found — run Cell 6 first for provenance data.")

# Same normalization as the main audit (Cell 2) — used as a fallback so legacy
# folders named by pub number only (no _familyID suffix) still match.
def _normalize_id_7(raw: str) -> str:
    pid = str(raw).strip().rstrip("/")
    pid = re.sub(r"PAFP$", "", pid)
    if re.match(r"^record_\d+$", pid):
        return pid
    if re.match(r"^USD\d", pid):
        pid = "SD" + pid[3:]
    us_pub = re.match(r"^(US)((?:19|20)\d{2})0*(\d+)[A-Za-z0-9]*$", pid)
    if us_pub:
        country, year, core = us_pub.groups()
        return f"{country}{year}_{core}"
    pid = re.sub(r"[A-Z][0-9]$", "", pid)
    pid = re.sub(r"[A-Z]$", "", pid)
    return pid

mem_col = "Simple Family Members"
norm_to_fam_7: dict[str, str] = {}
for _, row in df_xl.iterrows():
    pub = str(row.get(id_col, "")).strip()
    fam = str(row.get(fam_col, "")).strip()
    if not pub or not fam or pub.lower() == "nan" or fam.lower() == "nan":
        continue
    members_raw = str(row.get(mem_col, "")).strip()
    member_pubs = [m.strip() for m in re.split(r"[/\n]+", members_raw)
                   if m.strip() and m.strip().lower() != "nan"]
    for p in {pub} | set(member_pubs):
        norm_to_fam_7.setdefault(_normalize_id_7(p), fam)

# Build a map from family_id → folder path (folders named {patent_id}_{family_id};
# legacy pub-number-only folders are resolved via normalization)
_FAM_SUFFIX_RE_7 = re.compile(r"_(\d+)$")
fam_to_folder: dict[str, Path] = {}
for entry in sorted(raw_dir.iterdir()):
    if not entry.is_dir():
        continue
    m = _FAM_SUFFIX_RE_7.search(entry.name)
    fam = m.group(1) if m else norm_to_fam_7.get(_normalize_id_7(entry.name), "")
    if fam:
        fam_to_folder.setdefault(fam, entry)

def _col(df, *names):
    for n in names:
        if n in df.columns:
            return n
    return None

no_of_draw_col  = _col(df_xl, "No. of Drawings", "No of Drawings")
title_col       = _col(df_xl, "Title")
assignee_col    = _col(df_xl, "Assignee")
filing_col      = _col(df_xl, "Filing/Application Date")
fam_members_col = _col(df_xl, "Simple Family Members")

rows_exp = []
for _, row in df_xl.iterrows():
    pub = str(row.get(id_col, "")).strip()
    fam = str(row.get(fam_col, "")).strip()
    if not pub or pub.lower() == "nan":
        continue

    no_draw_raw = str(row[no_of_draw_col]).strip() if no_of_draw_col else ""
    try:
        no_draw = int(float(no_draw_raw)) if no_draw_raw and no_draw_raw.lower() != "nan" else None
    except ValueError:
        no_draw = None

    folder_path      = fam_to_folder.get(fam)
    folder_exists    = folder_path is not None
    actual_img_count = len(list(folder_path.glob("*.png"))) if folder_exists else 0

    count_matches = None
    if no_draw and no_draw > 0 and folder_exists:
        count_matches = actual_img_count == no_draw

    provenance_ok = prov_map.get(fam, "UNKNOWN") if fam else "UNKNOWN"

    rows_exp.append({
        "simple_family_id":      fam,
        "record_number":         pub,
        "matched_folder":        folder_path.name if folder_exists else "",
        "no_of_drawings":        no_draw if no_draw is not None else "",
        "title":                 str(row[title_col]).strip() if title_col else "",
        "assignee":              str(row[assignee_col]).strip() if assignee_col else "",
        "filing_date":           str(row[filing_col]).strip() if filing_col else "",
        "simple_family_members": str(row[fam_members_col]).strip() if fam_members_col else "",
        "folder_exists":         folder_exists,
        "actual_image_count":    actual_img_count,
        "count_matches":         "" if count_matches is None else count_matches,
        "provenance_ok":         provenance_ok,
    })

df_exp = pd.DataFrame(rows_exp)
df_exp.sort_values(by=["folder_exists", "simple_family_id"], ascending=[False, True], inplace=True)

out = data_dir / "patseer_expected_inventory.xlsx"
df_exp.to_excel(out, index=False)
print(f"PatSeer expected inventory → {out}")
print(f"  Total rows         : {len(df_exp)}")
print(f"  folder_exists=True : {df_exp['folder_exists'].sum()}")
print(f"  folder_exists=False: {(~df_exp['folder_exists']).sum()}")

/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


PatSeer expected inventory → /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/patseer_expected_inventory.xlsx
  Total rows         : 1639
  folder_exists=True : 1592
  folder_exists=False: 47


In [10]:
# ── Cell 8: Cross-reference summary ──────────────────────────────────────────
import sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
_cl_path  = repo_root / "src" / "config_loader.py"
_cl_spec  = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod   = importlib.util.module_from_spec(_cl_spec)
_cl_spec.loader.exec_module(_cl_mod)
cfg = _cl_mod.load_config()

import pandas as pd

raw_dir  = Path(cfg["paths"]["raw_images"])
data_dir = Path(cfg["paths"]["data"])

inv_path = data_dir / "disk_inventory.xlsx"
exp_path = data_dir / "patseer_expected_inventory.xlsx"
if not inv_path.exists() or not exp_path.exists():
    print("⚠  Run Cells 6 and 7 first to generate the inventory files.")
else:
    df_inv = pd.read_excel(inv_path, dtype=str)
    df_exp = pd.read_excel(exp_path, dtype=str)

    df_xl     = pd.read_excel(cfg["paths"]["patseer_excel"], dtype=str)
    fam_col   = "Simple Family ID"
    total_xl  = len(df_xl)
    total_fam = df_xl[fam_col].dropna().str.strip().nunique()

    total_disk = len(df_inv)

    present = df_exp[df_exp["folder_exists"].astype(str).str.lower() == "true"].copy()

    prov_ok      = present[present["provenance_ok"] == "True"]
    prov_fail    = present[present["provenance_ok"] == "False"]
    prov_unknown = present[present["provenance_ok"] == "UNKNOWN"]

    def _to_int(v):
        try:
            return int(float(str(v)))
        except (ValueError, TypeError):
            return None

    prov_ok_with_draw = prov_ok[prov_ok["no_of_drawings"].astype(str).str.match(r"^\d+$")].copy()
    prov_ok_with_draw["_exp"]   = prov_ok_with_draw["no_of_drawings"].apply(_to_int)
    prov_ok_with_draw["_act"]   = prov_ok_with_draw["actual_image_count"].apply(_to_int)
    prov_ok_with_draw["_delta"] = (prov_ok_with_draw["_act"] - prov_ok_with_draw["_exp"]).abs()

    count_ok        = int((prov_ok_with_draw["_delta"] == 0).sum())
    count_mismatch  = int((prov_ok_with_draw["_delta"] > 0).sum())
    prov_ok_no_draw = len(prov_ok) - len(prov_ok_with_draw)
    fully_verified  = count_ok + prov_ok_no_draw

    # Extra folders: disk family_id not in Excel
    excel_families = set(df_xl[fam_col].dropna().str.strip())
    extra_folders  = int((~df_inv["family_id"].isin(excel_families)).sum())

    n_missing = int((~df_exp["folder_exists"].astype(str).str.lower().eq("true")).sum())

    print("══════════════════════════════════════════════════════════")
    print("Dataset Integrity Report")
    print("══════════════════════════════════════════════════════════")
    print(f"  Total patents in Excel ({total_fam} families)  : {total_xl}")
    print(f"  Folders on disk                         : {total_disk:>4d}")
    print(f"  ─────────────────────────────────────────")
    print(f"  ✓ Fully verified                        : {fully_verified:>4d}   (folder exists + provenance OK)")
    print(f"  ✓ Provenance OK but count mismatch      : {count_mismatch:>4d}   (images present but wrong number)")
    print(f"  ✗ Provenance FAIL                       : {len(prov_fail):>4d}   (wrong patent's images in folder)")
    print(f"  ? Provenance UNKNOWN                    : {len(prov_unknown):>4d}   (bare fig_NN names — download interrupted)")
    print(f"  ✗ Missing from disk                     : {n_missing:>4d}")
    print(f"  ? Extra folders (not in Excel)          : {extra_folders:>4d}")
    print("══════════════════════════════════════════════════════════")

    if count_mismatch > 0:
        worst = (prov_ok_with_draw[prov_ok_with_draw["_delta"] > 0]
                 .sort_values("_delta", ascending=False)
                 .head(10)[["simple_family_id", "_exp", "_act", "_delta"]]
                 .rename(columns={"simple_family_id": "family_id",
                                  "_exp": "expected", "_act": "actual", "_delta": "delta"}))
        print("\nTop count mismatches:")
        print(worst.to_string(index=False))

    print()
    if len(prov_fail) == 0:
        msg = "✓ DATA INTEGRITY: CLEAN — no provenance failures"
        if len(prov_unknown):
            msg += f"  ({len(prov_unknown)} bare fig_NN folders — download interrupted, see Needs Redownload)"
        print(msg)
    else:
        print("✗ DATA INTEGRITY: ISSUES FOUND — see disk_inventory.xlsx for details")

/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


══════════════════════════════════════════════════════════
Dataset Integrity Report
══════════════════════════════════════════════════════════
  Total patents in Excel (1639 families)  : 1639
  Folders on disk                         : 1600
  ─────────────────────────────────────────
  ✓ Fully verified                        : 1592   (folder exists + provenance OK)
  ✓ Provenance OK but count mismatch      :    0   (images present but wrong number)
  ✗ Provenance FAIL                       :    0   (wrong patent's images in folder)
  ? Provenance UNKNOWN                    :    0   (bare fig_NN names — download interrupted)
  ✗ Missing from disk                     :   47
  ? Extra folders (not in Excel)          :    5
══════════════════════════════════════════════════════════

✓ DATA INTEGRITY: CLEAN — no provenance failures


## Cell 9 — Provenance deep-dive

Investigates the two categories that Cell 8 cannot fully verify:

| Category | What we do here |
|----------|-----------------|
| **Provenance FAIL** | Show exactly which pub numbers appear in the files vs. what the folder expects |
| **Provenance UNKNOWN** | All-`fig_NN` fallback folders. Cross-check against the "Needs Redownload" list — every UNKNOWN folder should be there; any that aren't are genuine gaps |

After the Cell 6 fix, UNKNOWN folders are **only** all-`fig_NN` folders (400-error fallbacks). Folders with `imgNNNN` / numeric / PAFP files are now correctly marked `True`.

**Outputs written to `data/`:**
- `provenance_fail_detail.csv` — folders with wrong-patent images (for 00a cleanup)
- `provenance_unknown_not_flagged.csv` — any fig_NN folders not yet in Needs Redownload (genuine gaps)

In [11]:
# ── Cell 9: Provenance deep-dive ─────────────────────────────────────────────
# Requires: Cell 6 (disk_inventory.xlsx) and Cell 4 (needs_redownload list)
# Re-loads everything from files so this cell is safe to run standalone.
import re, csv, sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
_cl_path  = repo_root / "src" / "config_loader.py"
_cl_spec  = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod   = importlib.util.module_from_spec(_cl_spec)
_cl_spec.loader.exec_module(_cl_mod)
cfg = _cl_mod.load_config()

import pandas as pd

raw_dir  = Path(cfg["paths"]["raw_images"])
data_dir = Path(cfg["paths"]["data"])
data_dir.mkdir(parents=True, exist_ok=True)

inv_path = data_dir / "disk_inventory.xlsx"
aud_path = Path(repo_root) / "notebooks" / "outputs" / "1639_DS_audit.xlsx"

if not inv_path.exists():
    print("⚠  disk_inventory.xlsx not found — run Cell 6 first.")
    raise SystemExit

df_inv = pd.read_excel(inv_path, dtype=str)

# ── Load the "Needs Redownload" list from the audit Excel ────────────────────
needs_redownload_folders: set[str] = set()
if aud_path.exists():
    import openpyxl
    _wb_sheets = openpyxl.load_workbook(aud_path, read_only=True).sheetnames
    if "Needs Redownload" in _wb_sheets:
        df_nr = pd.read_excel(aud_path, sheet_name="Needs Redownload", dtype=str)
        needs_redownload_folders = set(df_nr["folder_name"].dropna().str.strip())
    else:
        print("ℹ  No 'Needs Redownload' sheet in audit Excel (0 folders to redownload — this is fine).")
else:
    print("⚠  1639_DS_audit.xlsx not found — run Cells 3+4 first for redownload cross-check.")

# ── Same normalization used by Cells 2 & 6 ───────────────────────────────────
def _norm(raw: str) -> str:
    pid = str(raw).strip().rstrip("/")
    pid = re.sub(r"PAFP$", "", pid)
    if re.match(r"^record_\d+$", pid): return pid
    if re.match(r"^USD\d", pid): pid = "SD" + pid[3:]
    us = re.match(r"^(US)((?:19|20)\d{2})0*(\d+)[A-Za-z0-9]*$", pid)
    if us:
        country, year, core = us.groups()
        return f"{country}{year}_{core}"
    pid = re.sub(r"[A-Z][0-9]$", "", pid)
    pid = re.sub(r"[A-Z]$", "", pid)
    return pid

_PUB_RE = re.compile(r"^([A-Z]{2}[0-9A-Z]+)_")

# Column names used in both output CSVs — defined once so Cell 1b can rely on them
_FAIL_FIELDS    = ["folder_name","family_id","expected_pub","pub_prefixes_found",
                   "image_count","in_redownload_list","action"]
_UNKNOWN_FIELDS = ["folder_name","family_id","image_count","expected_pub"]

# Folders that are part of the pipeline infrastructure — never flag as gaps
_INFRA_FOLDERS = {"_quarantine", "missing_batch"}

# ══════════════════════════════════════════════════════════════════════════════
# PART A — Provenance FAIL folders
# ══════════════════════════════════════════════════════════════════════════════
fail_rows = df_inv[df_inv["provenance_ok"] == "False"].copy()

print(f"{'═'*60}")
print(f"PART A — Provenance FAIL  ({len(fail_rows)} folders)")
print(f"{'═'*60}")
print("These folders contain images whose pub-number prefix does NOT")
print("match any known member of the expected patent family.\n")

fail_detail = []
for _, row in fail_rows.iterrows():
    folder_name  = row["folder_name"]
    family_id    = row["family_id"]
    expected_pub = row["expected_pub_num"]
    folder_path  = raw_dir / folder_name

    actual_prefixes: set[str] = set()
    all_filenames: list[str]  = []
    if folder_path.exists():
        for f in sorted(folder_path.iterdir()):
            if f.suffix.lower() == ".png":
                all_filenames.append(f.name)
                m = _PUB_RE.match(f.name)
                if m:
                    actual_prefixes.add(m.group(1))

    actual_str    = " | ".join(sorted(actual_prefixes)) if actual_prefixes else "(no recognisable prefix)"
    files_str     = " | ".join(all_filenames[:5]) + (" ..." if len(all_filenames) > 5 else "")
    in_redownload = folder_name in needs_redownload_folders

    print(f"  Folder   : {folder_name}")
    print(f"  Family ID: {family_id}")
    print(f"  Expected : {expected_pub}  (norm → {_norm(expected_pub)})")
    print(f"  Found    : {actual_str}")
    print(f"  Files    : {files_str}")
    print(f"  In 'Needs Redownload' list: {'YES ✓' if in_redownload else 'NO  ← needs manual action'}")
    print(f"  → ACTION: delete folder and re-download via 00a")
    print()

    fail_detail.append({
        "folder_name":        folder_name,
        "family_id":          family_id,
        "expected_pub":       expected_pub,
        "pub_prefixes_found": actual_str,
        "image_count":        row["image_count"],
        "in_redownload_list": in_redownload,
        "action":             "DELETE and re-download",
    })

# Always write the header row so pd.read_csv() never hits EmptyDataError
fail_csv = data_dir / "provenance_fail_detail.csv"
with fail_csv.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=_FAIL_FIELDS)
    writer.writeheader()
    writer.writerows(fail_detail)
if fail_detail:
    print(f"Fail detail → {fail_csv}\n")
else:
    print(f"✓ No provenance FAIL folders — header-only CSV written to {fail_csv}\n")

# ══════════════════════════════════════════════════════════════════════════════
# PART B — Provenance UNKNOWN folders (fig_NN fallbacks)
# ══════════════════════════════════════════════════════════════════════════════
unknown_rows = df_inv[df_inv["provenance_ok"] == "UNKNOWN"].copy()

print(f"{'═'*60}")
print(f"PART B — Provenance UNKNOWN  ({len(unknown_rows)} folders)")
print(f"{'═'*60}")
print("These folders contain only fig_NN.png fallback images —")
print("no pub number is embedded in the filename, so provenance")
print("cannot be verified from filename alone.\n")
print("Cross-checking against 'Needs Redownload' list...\n")

already_flagged  = []
not_flagged      = []

for _, row in unknown_rows.iterrows():
    folder_name = row["folder_name"]
    # Skip infrastructure folders that are never patent downloads
    if folder_name in _INFRA_FOLDERS or folder_name.startswith("_"):
        continue
    if folder_name in needs_redownload_folders:
        already_flagged.append(folder_name)
    else:
        not_flagged.append({
            "folder_name":  folder_name,
            "family_id":    row["family_id"],
            "image_count":  row["image_count"],
            "expected_pub": row["expected_pub_num"],
        })

print(f"  ✓ Already in 'Needs Redownload': {len(already_flagged)}")
print(f"  ✗ NOT in 'Needs Redownload'    : {len(not_flagged)}  {'← these slipped through!' if not_flagged else '✓'}\n")

if not_flagged:
    print("  Folders with UNKNOWN provenance NOT flagged for redownload:")
    for r in not_flagged:
        print(f"    {r['folder_name']}  (fam={r['family_id']}, {r['image_count']} imgs, expected={r['expected_pub']})")

# Always write the header row so pd.read_csv() never hits EmptyDataError
gap_csv = data_dir / "provenance_unknown_not_flagged.csv"
with gap_csv.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=_UNKNOWN_FIELDS)
    writer.writeheader()
    writer.writerows(not_flagged)
if not_flagged:
    print(f"\n  → Written to {gap_csv}")
    print("    Add these to your 00a re-download run (delete folder first).")
else:
    print(f"  ✓ All UNKNOWN folders are already flagged — no gaps.")
    print(f"  → Header-only CSV written to {gap_csv}")

# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print("PROVENANCE DEEP-DIVE SUMMARY")
print(f"{'═'*60}")
print(f"  Provenance FAIL    : {len(fail_rows):>3d}  → delete + re-download (see provenance_fail_detail.csv)")
print(f"  UNKNOWN flagged    : {len(already_flagged):>3d}  → already in Needs Redownload list ✓")
print(f"  UNKNOWN NOT flagged: {len(not_flagged):>3d}  → {'see provenance_unknown_not_flagged.csv' if not_flagged else 'none ✓'}")
total_action = len(fail_rows) + len(not_flagged)
print(f"  ─────────────────────────────────────────")
print(f"  Total extra folders to delete+re-download: {total_action}")


ℹ  No 'Needs Redownload' sheet in audit Excel (0 folders to redownload — this is fine).
════════════════════════════════════════════════════════════
PART A — Provenance FAIL  (0 folders)
════════════════════════════════════════════════════════════
These folders contain images whose pub-number prefix does NOT
match any known member of the expected patent family.

✓ No provenance FAIL folders — header-only CSV written to /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/provenance_fail_detail.csv

════════════════════════════════════════════════════════════
PART B — Provenance UNKNOWN  (5 folders)
════════════════════════════════════════════════════════════
These folders contain only fig_NN.png fallback images —
no pub number is embedded in the filename, so provenance
cannot be verified from filename alone.

Cross-checking against 'Needs Redownload' list...

  ✓ Already in 'Needs Redownload': 0
  ✗ NOT in 'Needs Redownload'    : 3  ← these s

In [12]:
# ── Cell 10: Rename legacy folders + quarantine truly-extra ones ─────────────
# Legacy folders use old naming (e.g. CA3096221) instead of {pub}_{fam_id}.
# This cell:
#   1. Resolves each legacy folder to its Excel family via pub-number normalisation
#   2. DRY-RUN: prints every rename without touching disk
#   3. After confirmation (set DRY_RUN=False), applies renames in-place
#   4. Moves the 3 truly-extra folders to _quarantine/
#   5. Rewrites folder_naming_issues.csv to reflect the clean state
import re, csv, sys, shutil, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
_cl_path  = repo_root / "src" / "config_loader.py"
_cl_spec  = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod   = importlib.util.module_from_spec(_cl_spec)
_cl_spec.loader.exec_module(_cl_mod)
cfg = _cl_mod.load_config()

import pandas as pd

raw_dir  = Path(cfg["paths"]["raw_images"])
data_dir = Path(cfg["paths"]["data"])
data_dir.mkdir(parents=True, exist_ok=True)

# ── Set to False to apply renames for real ───────────────────────────────────
DRY_RUN = True

# ── Infrastructure folders — never touch ─────────────────────────────────────
_INFRA = {"_quarantine", "missing_batch"}

# ── Normalisation (same as rest of notebook) ──────────────────────────────────
def _norm(raw: str) -> str:
    pid = str(raw).strip().rstrip("/")
    pid = re.sub(r"PAFP$", "", pid)
    if re.match(r"^record_\d+$", pid): return pid
    if re.match(r"^USD\d", pid): pid = "SD" + pid[3:]
    us = re.match(r"^(US)((?:19|20)\d{2})0*(\d+)[A-Za-z0-9]*$", pid)
    if us:
        country, year, core = us.groups()
        return f"{country}{year}_{core}"
    pid = re.sub(r"[A-Z][0-9]$", "", pid)
    pid = re.sub(r"[A-Z]$", "", pid)
    return pid

_FAM_SUFFIX_RE = re.compile(r"_(\d+)$")

# ── Build norm → family_id map from Excel ─────────────────────────────────────
df_xl   = pd.read_excel(cfg["paths"]["patseer_excel"], dtype=str)
id_col  = "Record Number"
fam_col = "Simple Family ID"
mem_col = "Simple Family Members"

norm_to_fam: dict[str, str] = {}
fam_to_pub:  dict[str, str] = {}

for _, row in df_xl.iterrows():
    pub = str(row.get(id_col, "")).strip()
    fam = str(row.get(fam_col, "")).strip()
    if not pub or not fam or pub.lower() == "nan" or fam.lower() == "nan":
        continue
    fam_to_pub.setdefault(fam, pub)
    members_raw = str(row.get(mem_col, "")).strip()
    member_pubs = [m.strip() for m in re.split(r"[/\n]+", members_raw)
                   if m.strip() and m.strip().lower() != "nan"]
    for p in {pub} | set(member_pubs):
        norm_to_fam.setdefault(_norm(p), fam)

excel_fam_ids = set(fam_to_pub.keys())

# ── Find all legacy folders (no numeric family_id suffix) ─────────────────────
legacy = [
    e for e in sorted(raw_dir.iterdir())
    if e.is_dir() and not _FAM_SUFFIX_RE.search(e.name) and e.name not in _INFRA
]

to_rename    = []   # (folder_path, new_name, family_id)
to_quarantine = []  # folder_path — truly not in Excel

for folder in legacy:
    fam = norm_to_fam.get(_norm(folder.name))
    if fam:
        pub          = fam_to_pub.get(fam, folder.name)
        new_name     = f"{pub}_{fam}"
        to_rename.append((folder, new_name, fam))
    else:
        to_quarantine.append(folder)

print(f"Legacy folders found        : {len(legacy)}")
print(f"  → Resolvable (to rename)  : {len(to_rename)}")
print(f"  → Not in Excel (quarantine): {len(to_quarantine)}")
print()

# ── DRY-RUN preview ───────────────────────────────────────────────────────────
if DRY_RUN:
    print(f"{'═'*70}")
    print(f"DRY RUN — no files will be changed (set DRY_RUN=False to apply)")
    print(f"{'═'*70}")
    print()

print(f"── Renames ({len(to_rename)}) {'[DRY RUN]' if DRY_RUN else '[APPLYING]'} ──")
rename_log = []
for folder, new_name, fam in to_rename:
    new_path = raw_dir / new_name
    conflict = new_path.exists() and new_path != folder
    status   = "CONFLICT — target exists" if conflict else ("renamed" if not DRY_RUN else "would rename")
    print(f"  {folder.name:<45} → {new_name}  [{status}]")
    rename_log.append({
        "original_name":    folder.name,
        "new_name":         new_name if not conflict else "",
        "simple_family_id": fam,
        "status":           status,
    })

print()
print(f"── Quarantine ({len(to_quarantine)}) {'[DRY RUN]' if DRY_RUN else '[APPLYING]'} ──")
quarantine_dir = raw_dir / "_quarantine"
for folder in to_quarantine:
    imgs   = len(list(folder.glob("*.png")))
    action = "would move" if DRY_RUN else "moved"
    print(f"  {folder.name:<45} ({imgs} imgs) → _quarantine/  [{action}]")

# ── Apply (only when DRY_RUN=False) ──────────────────────────────────────────
if not DRY_RUN:
    quarantine_dir.mkdir(exist_ok=True)
    applied = skipped = 0
    for folder, new_name, fam in to_rename:
        new_path = raw_dir / new_name
        if new_path.exists() and new_path != folder:
            print(f"  SKIP conflict: {folder.name} → {new_name}")
            skipped += 1
            continue
        folder.rename(new_path)
        applied += 1
    print(f"\nRenamed: {applied}  Skipped (conflict): {skipped}")

    for folder in to_quarantine:
        dest = quarantine_dir / folder.name
        shutil.move(str(folder), str(dest))
        print(f"  Quarantined: {folder.name}")

    # Rewrite folder_naming_issues.csv
    remaining_issues = []
    for entry in sorted(raw_dir.iterdir()):
        if not entry.is_dir(): continue
        if _FAM_SUFFIX_RE.search(entry.name): continue
        if entry.name in _INFRA: continue
        remaining_issues.append({"folder": entry.name, "issue": "no numeric family_id suffix"})
    issues_path = data_dir / "folder_naming_issues.csv"
    with issues_path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["folder", "issue"])
        w.writeheader()
        w.writerows(remaining_issues)
    print(f"\nfolder_naming_issues.csv updated: {len(remaining_issues)} remaining issues")

    # Overwrite folder_rename_log.csv
    log_path = data_dir / "folder_rename_log.csv"
    with log_path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["original_name", "new_name", "simple_family_id", "status"])
        w.writeheader()
        w.writerows(rename_log)
    print(f"folder_rename_log.csv written: {len(rename_log)} entries")

print()
if DRY_RUN:
    print("─" * 70)
    print("Set DRY_RUN = False and re-run this cell to apply the changes.")
else:
    print("✓ Done. Re-run Cells 2–5 to verify the audit reflects the new names.")


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Legacy folders found        : 183
  → Resolvable (to rename)  : 180
  → Not in Excel (quarantine): 3

══════════════════════════════════════════════════════════════════════
DRY RUN — no files will be changed (set DRY_RUN=False to apply)
══════════════════════════════════════════════════════════════════════

── Renames (180) [DRY RUN] ──
  AU2020100605A4                                → AU2020100605A4_70776151  [would rename]
  CA2947683A1                                   → CA2947683A1_62068329  [would rename]
  CA2958445A1                                   → CA2958445A1_63166000  [would rename]
  CA3096221                                     → CA3096221A1_81206876  [would rename]
  CN112478154A                                  → CN112478154A_74940243  [would rename]
  CN114435591A                                  → CN114435591A_81374329  [would rename]
  CN117842351A                                  → CN117842351A_90542130  [would rename]
  CN119218433A                                